# Paper-Inspired Task A Model — CodeBERT + 14 Static Features

**Paper:** Cotroneo, Improta, Liguori (2025) — *Human-Written vs. AI-Generated Code: A Large-Scale Study of Defects, Vulnerabilities, and Complexity* (arXiv:2508.21634)

### Architecture
- **Backbone:** `microsoft/codebert-base` → [CLS] embedding (768-d)
- **Features:** 14 paper-motivated static features, extracted **per language** via `lizard` + `tiktoken` + regex
- **Head:** `[CLS](768) ⊕ features(14)  → 256 → 128 → 64 → 2` (GELU + LayerNorm + Dropout)
- **Loss:** Weighted Focal Loss (gamma=2)
- **Optimizer:** LLRD + cosine warmup

### Compute: Kaggle T4 (16 GB VRAM)
- `SAMPLE_SIZE = 100_000` — stratified


In [ ]:
!pip install --quiet --upgrade transformers==4.45.0 huggingface_hub lizard tiktoken
print('Dependencies ready.')

In [ ]:
import os, re, math, hashlib, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from tqdm import tqdm

import lizard
import tiktoken

from datasets import Dataset
from transformers import (
    RobertaTokenizer, RobertaModel,
    TrainingArguments, Trainer,
    EarlyStoppingCallback, DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
)

os.environ['WANDB_DISABLED'] = 'true'
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

_ENC = tiktoken.get_encoding('cl100k_base')
print('Imports ready.')

In [ ]:
DATA_ROOT     = ("/kaggle/input/datasets/daniilor/semeval-2026-task13/"
                 "SemEval-2026-Task13/task_a/")
TRAIN_PARQUET = DATA_ROOT + "task_a_training_set_1.parquet"
VAL_PARQUET   = DATA_ROOT + "task_a_validation_set.parquet"
TEST_PARQUET  = DATA_ROOT + "task_a_test_set_sample.parquet"
OUTPUT_DIR    = "/kaggle/working/paper_inspired_model"

MODEL_NAME        = "microsoft/codebert-base"
MAX_LENGTH        = 512
SAMPLE_SIZE       = 100_000
VAL_SAMPLE_SIZE   = 20_000
RANDOM_SEED       = 42

NUM_EPOCHS        = 8
BATCH_SIZE        = 8
GRAD_ACCUM        = 4
BASE_LR           = 2e-5
HEAD_LR           = 5e-4
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.06
GRAD_CLIP         = 1.0
LLRD_FACTOR       = 0.9
DROPOUT           = 0.2
FOCAL_GAMMA       = 2.0
NUM_CODE_FEATURES = 14

In [ ]:
PLACEHOLDERS = {"unk", "na", "n/a", "-", "?", "none", "null", "nan"}
ENCODING_RE  = re.compile(r"Ã.|â€™|â€œ|â€|�|\?\?\?")

def _normalize_ws(s): return s.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
def _sha1(s): return s.map(lambda x: hashlib.sha1(x.encode("utf-8", "ignore")).hexdigest())

def clean_df(df, text_col="code", label_col="label"):
    raw = df[text_col]
    bad = (raw.isna() | raw.fillna("").astype(str).eq("") |
           raw.fillna("").astype(str).str.fullmatch(r"\s*") |
           raw.fillna("").astype(str).str.strip().str.lower().isin(PLACEHOLDERS))
    df = df[~bad].reset_index(drop=True)
    t = _normalize_ws(df[text_col]); df = df[~t.str.contains(ENCODING_RE)].reset_index(drop=True)
    t = _normalize_ws(df[text_col]); h = _sha1(t); df = df[~h.duplicated(keep="first")].reset_index(drop=True)
    return df

print("Loading data...")
raw_df = pd.read_parquet(TRAIN_PARQUET)
raw_df["label"] = raw_df["label"].astype(int)
train_full = clean_df(raw_df)

lang_col   = next((c for c in train_full.columns if c.lower() in ("language", "lang")), None)
domain_col = next((c for c in train_full.columns if c.lower() in ("domain", "task_type")), None)

In [ ]:
def stratified_sample(df, n, seed, group_cols):
    cols = [c for c in group_cols if c and c in df.columns]
    if not cols: return df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)
    key = df[cols].astype(str).agg("_".join, axis=1)
    counts = key.value_counts()
    parts = []
    for k, cnt in counts.items():
        take = min(max(1, round(cnt / len(df) * n)), cnt)
        parts.append(df[key == k].sample(n=take, random_state=seed))
    res = pd.concat(parts).reset_index(drop=True)
    if len(res) < n:
        rem = df[~df.index.isin(res.index)]
        res = pd.concat([res, rem.sample(n=min(n - len(res), len(rem)), random_state=seed)]).reset_index(drop=True)
    return res.head(n)

train_df = stratified_sample(train_full, SAMPLE_SIZE, RANDOM_SEED, ["label", lang_col, domain_col])
val_raw = clean_df(pd.read_parquet(VAL_PARQUET).assign(label=lambda x: x["label"].astype(int)))
val_df = stratified_sample(val_raw, VAL_SAMPLE_SIZE, RANDOM_SEED, ["label", lang_col, domain_col])

In [ ]:
_LANG_EXT = {
    "python": "_.py", "java": "_.java", "c++": "_.cpp", "cpp": "_.cpp", "c": "_.c",
    "c#": "_.cs", "csharp": "_.cs", "javascript": "_.js", "js": "_.js", "go": "_.go",
    "php": "_.php", "ruby": "_.rb", "swift": "_.swift", "kotlin": "_.kt",
    "typescript": "_.ts", "rust": "_.rs", "scala": "_.scala",
}
def _lang_file(lang): return _LANG_EXT.get(str(lang).strip().lower(), "_.py")

_RE_DEBUG   = re.compile(r"\bprint\s*\(|System\.out\.print|console\.log\s*\(|fmt\.Print|\bprintln!\(", re.I)
_RE_SUBPROC = re.compile(r"subprocess\.|os\.system\s*\(|exec\.Command|Runtime\.getRuntime\(\)|ProcessBuilder|shell_exec\s*\(|popen\s*\(", re.I)
_RE_HARDCOD = re.compile(r"(?:password|passwd|secret|api_key|token|credential|auth)\s*=\s*['\"][^'\"]{4,}['\"]", re.I)
_RE_EXCEPT  = re.compile(r"\btry\b|\bcatch\b|\bexcept\b|\bfinally\b", re.I)
_RE_COMMENT = re.compile(r"^\s*(#|//|/\*|\*)", re.M)

def extract_paper_features(code: str, language: str = "python") -> list:
    if not code or not code.strip(): return [0.0] * 14
    # Truncate to avoid Regex/lizard hangs on minified files
    if len(code) > 10000:
        code = code[:10000]
    # Truncate to avoid Regex/lizard hangs on minified files
    if len(code) > 10000:
        code = code[:10000]
    lines = code.split("\n")
    non_empty = [l for l in lines if l.strip()]
    nloc = max(len(non_empty), 1)

    try:
        res = lizard.analyze_file.analyze_source_code(_lang_file(language), code)
        if res.function_list:
            fn = res.function_list[0]
            ccn, fnl, params = min(fn.cyclomatic_complexity / 10.0, 1.0), min(len(fn.name) / 30.0, 1.0), min(fn.parameter_count / 10.0, 1.0)
        else:
            ccn, fnl, params = min(getattr(res, "average_cyclomatic_complexity", 1) / 10.0, 1.0), 0.0, 0.0
    except: ccn, fnl, params = 0.1, 0.0, 0.0

    try:
        tokens = _ENC.encode(code)
        tc, ut = len(tokens), len(set(tokens))
    except: tc, ut = 1, 1

    tc_norm, ut_norm, ut_ratio = min(tc / 200.0, 1.0), min(ut / 300.0, 1.0), ut / max(tc, 1)

    c_rat = len(_RE_COMMENT.findall(code)) / max(len(lines), 1)
    b_rat = sum(1 for l in lines if not l.strip()) / max(len(lines), 1)
    lens = [len(l) for l in non_empty]
    ml = sum(lens) / len(lens) if lens else 0.0
    cv = min(math.sqrt(sum((x - ml)**2 for x in lens) / len(lens)) / ml / 2.0, 1.0) if len(lens) > 1 and ml > 0 else 0.0

    dd = min(len(_RE_DEBUG.findall(code)) / nloc, 1.0)
    sb = float(bool(_RE_SUBPROC.search(code)))
    hb = float(bool(_RE_HARDCOD.search(code)))
    er = min(len(_RE_EXCEPT.findall(code)) / nloc, 1.0)

    return [min(nloc / 50.0, 1.0), ccn, tc_norm, ut_ratio, ut_norm, fnl, params, c_rat, b_rat, dd, sb, hb, er, cv]

In [ ]:
def batch_extract(df):
    codes = df["code"].tolist()
    langs = df[lang_col].tolist() if lang_col else ["python"] * len(df)
    return [extract_paper_features(c, l) for c, l in tqdm(zip(codes, langs), total=len(codes))]

train_df = train_df.copy(); train_df["code_features"] = batch_extract(train_df)
val_df = val_df.copy(); val_df["code_features"] = batch_extract(val_df)

In [ ]:
class WeightedFocalLoss(nn.Module):
    def __init__(self, w, gamma=2.0):
        super().__init__(); self.register_buffer("w", w); self.gamma = gamma
    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, weight=self.w.to(logits.device), reduction="none")
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

cw = (lambda c: torch.tensor((c.sum() / (len(c) * c)).values, dtype=torch.float32))(train_df["label"].value_counts().sort_index())

class PaperInspiredCodeBERT(nn.Module):
    def __init__(self, m=MODEL_NAME, nl=2, nf=14, dp=0.2, w=cw, g=2.0):
        super().__init__()
        self.transformer = RobertaModel.from_pretrained(m)
        self.feat_norm = nn.LayerNorm(nf)
        self.head = nn.Sequential(
            nn.Linear(768+nf, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dp),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dp),
            nn.Linear(128, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(dp),
            nn.Linear(64, nl)
        )
        self.loss_fn = WeightedFocalLoss(w, gamma=g)
    def forward(self, input_ids=None, attention_mask=None, labels=None, code_features=None, **kw):
        cls = self.transformer(input_ids, attention_mask).last_hidden_state[:, 0]
        feat = self.feat_norm(code_features.float()) if code_features is not None else cls.new_zeros(cls.size(0), 14)
        logits = self.head(torch.cat([cls, feat], dim=-1))
        return SequenceClassifierOutput(loss=self.loss_fn(logits, labels) if labels is not None else None, logits=logits)

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
def to_hf(df):
    ds = Dataset.from_pandas(df[["code", "label", "code_features"]].reset_index(drop=True))
    return ds.map(lambda x: tokenizer(x["code"], truncation=True, max_length=MAX_LENGTH), batched=True, remove_columns=["code"]).rename_column("label", "labels")

train_ds, val_ds = to_hf(train_df), to_hf(val_df)

class PaperCollator:
    def __init__(self, tok): self.base = DataCollatorWithPadding(tokenizer=tok)
    def __call__(self, features):
        feats = [f.pop("code_features") for f in features] if "code_features" in features[0] else None
        batch = self.base(features)
        if feats is not None: batch["code_features"] = torch.tensor(feats, dtype=torch.float32)
        return batch
collator = PaperCollator(tokenizer)

In [ ]:
class PaperTrainer(Trainer):
    def create_optimizer_and_scheduler(self, num_training_steps):
        no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}
        N = self.model.config.num_hidden_layers
        params = []
        def add(p, lr, wd):
            wd_p, nd_p = [], []
            for n, p in p: (nd_p if any(k in n for k in no_decay) else wd_p).append(p)
            if wd_p: params.append({"params": wd_p, "lr": lr, "weight_decay": wd})
            if nd_p: params.append({"params": nd_p, "lr": lr, "weight_decay": 0.0})

        add(self.model.transformer.embeddings.named_parameters(), BASE_LR * LLRD_FACTOR**N, WEIGHT_DECAY)
        for i, layer in enumerate(self.model.transformer.encoder.layer): add(layer.named_parameters(), BASE_LR * LLRD_FACTOR**(N-i), WEIGHT_DECAY)
        add(list(self.model.head.named_parameters()) + list(self.model.feat_norm.named_parameters()), HEAD_LR, WEIGHT_DECAY)

        self.optimizer = torch.optim.AdamW(params)
        self.lr_scheduler = get_cosine_schedule_with_warmup(self.optimizer, int(num_training_steps * WARMUP_RATIO), num_training_steps)

def comp_metrics(eval_pred):
    p = np.argmax(eval_pred[0], axis=1); l = eval_pred[1]
    return {"macro_f1": precision_recall_fscore_support(l, p, average="macro", zero_division=0)[2]}

model = PaperInspiredCodeBERT().to(DEVICE)
args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM, weight_decay=WEIGHT_DECAY,
    eval_strategy="steps", eval_steps=1000, save_strategy="steps", save_steps=1000,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", fp16=True, report_to=[]
)

trainer = PaperTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer, data_collator=collator, compute_metrics=comp_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
trainer.train()
trainer.save_model(OUTPUT_DIR)

In [ ]:
@torch.no_grad()
def predict_on_df(model, tokenizer, df):
    model.eval(); c = df["code"].tolist()
    lcol = next((c for c in df.columns if c.lower() in ("language", "lang")), None)
    l = df[lcol].tolist() if lcol else ["python"] * len(df)
    preds = []
    for i in tqdm(range(0, len(c), 32)):
        bc, bl = c[i:i+32], l[i:i+32]
        enc = tokenizer(bc, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
        feat = torch.tensor([extract_paper_features(x, y) for x, y in zip(bc, bl)], dtype=torch.float32)
        preds.extend(model(input_ids=enc.input_ids.to(DEVICE), attention_mask=enc.attention_mask.to(DEVICE), code_features=feat.to(DEVICE)).logits.argmax(-1).cpu().tolist())
    return df.assign(prediction=preds)

def eval_cat(df):
    lcol = next((c for c in df.columns if c.lower() in ("language", "lang")), None)
    dcol = next((c for c in df.columns if c.lower() in ("domain", "task_type")), None)
    if "label" not in df: return
    df = df.copy(); l = df[lcol].str.lower() if lcol else None; d = df[dcol].str.lower() if dcol else None
    y_t, y_p = df["label"], df["prediction"]


In [ ]:
if os.path.exists(TEST_PARQUET):
    t_df = pd.read_parquet(TEST_PARQUET).dropna(subset=["code"])
    if "label" in t_df.columns:
        t_df["label"] = t_df["label"].astype(int)
        eval_cat(predict_on_df(model, tokenizer, t_df))
    sub = predict_on_df(model, tokenizer, t_df)
    pd.DataFrame({"id": sub.index, "label": sub["prediction"]}).to_csv("/kaggle/working/submission_task_a.csv", index=False)